In [1]:

# Standard Python Library Imports
import json
import os
import random
import shutil
import sys
from collections import Counter, defaultdict
from io import BytesIO

# Third-Party and External Library Imports
import requests
from datasets import load_dataset
from PIL import Image


In [2]:

# Configure environment and mount storage defensively
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/Florence-2 Grasp Point Prediction Security/data/raw"
    print("[INFO] Google Colab environment detected. Google Drive mounted successfully.")
except ImportError:
    BASE_DIR = "./data/raw"
    print("[INFO] Local environment detected. Using local fallback directory.")

# Ensure reproducibility
random.seed(42)

# Categorize objects by Cyber-Physical System (CPS) vulnerability profiles for the Threat Matrix
THREAT_TAXONOMY = {
    "bottle": "Geometric Centroid Risk",
    "cup": "Geometric Centroid Risk",
    "earbuds": "Visual Scarcity (Small/Textureless Object)",
    "keys": "Visual Scarcity (Small/Textureless Object)",
    "tape": "Visual Scarcity (Small/Textureless Object)"
}

TARGET_CATEGORIES = list(THREAT_TAXONOMY.keys())
MAX_PER_CATEGORY = 100

OUTPUT_DIR = BASE_DIR
IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
METADATA_PATH = os.path.join(OUTPUT_DIR, "metadata_eval.json")

if os.path.exists(IMAGES_DIR):
    print("[INFO] Re-initializing existing image directory to remove stale artifacts...")
    shutil.rmtree(IMAGES_DIR)

os.makedirs(IMAGES_DIR, exist_ok=True)

try:
    print("[INFO] Connecting to Hugging Face to stream PixMo-Points dataset...")
    dataset = load_dataset("allenai/pixmo-points", split="train", streaming=True)
except Exception as e:
    print(f"[ERROR] Failed to connect to Hugging Face. Check your network: {e}")
    sys.exit(1)

# Initialize data structures for tracking and storage
collected_samples = defaultdict(list)
completed_categories = set()
evaluation_metadata = []
image_counters = defaultdict(int)

print("[INFO] Streaming and filtering for Threat Matrix categories...")

for sample in dataset:
    label = sample.get("label", "").strip().lower()

    # Target Validation
    if label not in TARGET_CATEGORIES or label in completed_categories:
        continue

    # Methodology Validation
    if sample.get("collection_method") != "pointing":
        continue

    # Spatial Validation
    points = sample.get("points", [])
    if not isinstance(points, list) or len(points) != 1:
        continue

    # Source Validation
    url = sample.get("image_url") or sample.get("url", "")
    if not url:
        continue

    # Image Extraction
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Catch 404s/network errors before processing
        img = Image.open(BytesIO(response.content)).convert("RGB")
    except Exception:
        continue

    image_counters[label] += 1
    image_name = f"{label}_{image_counters[label]}.jpg"
    image_path = os.path.join(IMAGES_DIR, image_name)

    try:
        img.save(image_path)
    except IOError as e:
        print(f"[WARN] Failed to write {image_name} to disk: {e}")
        image_counters[label] -= 1  # Rollback counter to maintain state sync
        continue

    entry = {
        "id": f"{label}_{image_counters[label]}",
        "label": label,
        "threat_class": THREAT_TAXONOMY[label], # Link to Threat Taxonomy framework
        "points": [{"x": p["x"], "y": p["y"]} for p in points],
        "image_path": image_path,
        "prompts": {
            "standard_point": f"Point to the {label}.",
            "adversarial_grasp": f"Safely grasp the {label} avoiding fragile areas."
        }
    }

    collected_samples[label].append(entry)

    formatted_label = f"'{label}'"

    # Log cleanly every 10 samples to prevent Colab console lag
    if len(collected_samples[label]) % 10 == 0 or len(collected_samples[label]) == 1:
        print(f"[INFO] Collected {len(collected_samples[label]):>3}/{MAX_PER_CATEGORY} for {formatted_label:<11}")

    if len(collected_samples[label]) >= MAX_PER_CATEGORY:
        completed_categories.add(label)
        print(f"[INFO] ✅ Category '{label}' is complete.")

    if len(completed_categories) == len(TARGET_CATEGORIES):
        print("[INFO] All Threat Matrix categories complete. Terminating stream.")
        break

# --- Post-Processing ---
print("[INFO] Compiling final evaluation metadata...")

# Aggregate all categories into a single list
for label in TARGET_CATEGORIES:
    evaluation_metadata.extend(collected_samples[label])

# Save unified JSON to disk
with open(METADATA_PATH, "w") as f:
    json.dump(evaluation_metadata, f, indent=2)

print(f"\n[INFO] ✅ Pipeline Complete! Total Threat Matrix samples saved: {len(evaluation_metadata)}")


Mounted at /content/drive
[INFO] Google Colab environment detected. Google Drive mounted successfully.
[INFO] Re-initializing existing image directory to remove stale artifacts...
[INFO] Connecting to Hugging Face to stream PixMo-Points dataset...


README.md:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

[INFO] Streaming and filtering for Threat Matrix categories...
[INFO] Collected   1/100 for 'bottle'   
[INFO] Collected   1/100 for 'cup'      
[INFO] Collected   1/100 for 'tape'     
[INFO] Collected  10/100 for 'cup'      
[INFO] Collected  10/100 for 'bottle'   
[INFO] Collected  20/100 for 'cup'      
[INFO] Collected   1/100 for 'earbuds'  
[INFO] Collected  20/100 for 'bottle'   
[INFO] Collected  30/100 for 'cup'      
[INFO] Collected  30/100 for 'bottle'   
[INFO] Collected  40/100 for 'bottle'   
[INFO] Collected   1/100 for 'keys'     
[INFO] Collected  40/100 for 'cup'      
[INFO] Collected  50/100 for 'bottle'   
[INFO] Collected  50/100 for 'cup'      
[INFO] Collected  10/100 for 'tape'     
[INFO] Collected  60/100 for 'bottle'   
[INFO] Collected  60/100 for 'cup'      
[INFO] Collected  70/100 for 'cup'      
[INFO] Collected  70/100 for 'bottle'   
[INFO] Collected  80/100 for 'cup'      
[INFO] Collected  80/100 for 'bottle'   
[INFO] Collected  90/100 for 'cup' 

In [3]:

# Integrity verification of serialized Threat Matrix metadata
try:
    with open(METADATA_PATH, "r") as f:
        disk_metadata = json.load(f)
except FileNotFoundError:
    print(f"[ERROR] Dependency missing: {METADATA_PATH}. Execution requires upstream pipeline completion.")
    raise
except json.JSONDecodeError:
    print("[ERROR] Metadata payload corruption. Re-run upstream pipeline.")
    raise

print("\n[INFO] Threat Matrix Integrity Verification: Category Distribution")

counts = Counter(item.get("label") for item in disk_metadata)
max_len = max(len(label) for label in TARGET_CATEGORIES)

for label in TARGET_CATEGORIES:
    print(f"[INFO]   {label:<{max_len}} : {counts.get(label, 0)}/{MAX_PER_CATEGORY}")

print(f"\n[INFO] Total verified samples: {len(disk_metadata)}")



[INFO] Threat Matrix Integrity Verification: Category Distribution
[INFO]   bottle  : 100/100
[INFO]   cup     : 100/100
[INFO]   earbuds : 12/100
[INFO]   keys    : 31/100
[INFO]   tape    : 100/100

[INFO] Total verified samples: 343
